# 👑 GeoMAS - Telemetry Analysis & Insights

Questo notebook si connette al database della telemetria (`data/simulation_metrics.duckdb`) per estrarre le metriche globali, nazionali e relazionali salvate dall'engine.
Ti permette di tracciare andamenti continui (Time-Series) per la tua Tesi, confrontando scenari e valutando la Coerenza e il livello di Inganno (Deception) degli LLM.

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Configurazione Stile
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 7)

DB_PATH = 'simulation_metrics.duckdb'

# Utility per connettersi senza bloccare il file
def query_db(query, params=None):
    if params:
        # FIX: Converte i tipi numpy (es. int32) in tipi Python standard per evitare errori DuckDB
        params = [p.item() if hasattr(p, 'item') else p for p in params]
    with duckdb.connect(DB_PATH, read_only=True) as conn:
        return conn.execute(query, params or []).df()

## 1. Elenco delle Simulazioni
Vediamo quali run sono state registrate nel DB.

In [ ]:
simulations = query_db("SELECT DISTINCT simulation_id FROM metrics_global ORDER BY simulation_id")
print("Simulazioni Disponibili:")
display(simulations)

# Imposta l'ID della simulazione che vuoi analizzare in profondità
TARGET_SIM_ID = simulations['simulation_id'].iloc[-1] if not simulations.empty else None
print(f"\nAnalizzando Simulazione ID: {TARGET_SIM_ID}")

## 2. Metriche Globali (RQ1 & RQ4)
Andamento aggregato del mondo per valutare la stabilità e la disonestà sistemica.

In [ ]:
global_df = query_db("SELECT * FROM metrics_global WHERE simulation_id = ? ORDER BY turn", [TARGET_SIM_ID])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico 1: Deception & Coherence Media Mondiale
sns.lineplot(data=global_df, x='turn', y='global_deception_avg', ax=axes[0], label='Deception Media', color='red', linewidth=2)
sns.lineplot(data=global_df, x='turn', y='global_coherence_avg', ax=axes[0], label='Coherence Media', color='green', linewidth=2)
axes[0].set_title("Stabilità degli Agenti (Mondo)")
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_ylabel("Score (0-1)")

# Grafico 2: Militarizzazione vs Commercio
sns.lineplot(data=global_df, x='turn', y='global_trade_volume', ax=axes[1], label='Trade Volume', color='blue')
sns.lineplot(data=global_df, x='turn', y='units_created', ax=axes[1], label='Military Spending', color='orange')
axes[1].set_title("Guns vs Butter (Spesa Globale)")
axes[1].set_ylabel("Risorse")

plt.tight_layout()
plt.show()

## 3. Metriche Nazionali Grantulari (RQ3)
Confrontiamo come le diverse strategie globali adottate dalle nazioni abbiano impattato il loro livello di sincerità e la loro potenza.

In [ ]:
nation_df = query_db("SELECT * FROM metrics_nation WHERE simulation_id = ? ORDER BY turn, nation_id", [TARGET_SIM_ID])

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Deception Score per Nazione
sns.lineplot(data=nation_df, x='turn', y='deception_overall', hue='nation_id', ax=axes[0], linewidth=2)
axes[0].set_title("Deception Score per Nazione (Tasso di bugie diplomatiche/militari)")
axes[0].set_ylim(-0.1, 1.1)

# Power Projection per Nazione
sns.lineplot(data=nation_df, x='turn', y='power_projection', hue='nation_id', ax=axes[1], linewidth=2)
axes[1].set_title("Evoluzione del Potere (Power Projection)")

plt.tight_layout()
plt.show()

## 4. Soddisfazione Pubblica (RQ1)
Impatto delle guerre (o di scenari estremi) sulla popolazione.

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=nation_df, x='turn', y='public_satisfaction', hue='nation_id', linewidth=2)
plt.axhline(25, color='red', linestyle='--', label='Soglia Civil Unrest')
plt.title("Andamento della Soddisfazione Pubblica nel mondo")
plt.ylabel("Soddisfazione (0-100)")
plt.legend()
plt.show()

## 5. Network Analysis delle Relazioni Diplomatiche
Visualizziamo il grafo di Trust all'ultimo turno registrato per capire i blocchi di potere.

In [ ]:
# Prendiamo i dati dell'ultimo turno della simulazione target
max_turn = global_df['turn'].max() if not global_df.empty else 0
trust_df = query_db("SELECT * FROM metrics_trust WHERE simulation_id = ? AND turn = ?", [TARGET_SIM_ID, max_turn])

if not trust_df.empty:
    import matplotlib.lines as mlines
    plt.figure(figsize=(12, 10))
    G = nx.DiGraph()
    
    # Mappa dei colori e degli stili per le relazioni (Sincronizzato con GeoMAS schemas)
    REL_MAP = {
        'MUTUAL_DEFENSE': {'color': '#006400', 'style': 'solid', 'label': 'Mutual Defense (Alliance)'},
        'NON_AGGRESSION': {'color': '#32CD32', 'style': 'dashed', 'label': 'Non-Aggression Pact'},
        'WAR':            {'color': '#FF0000', 'style': 'solid', 'label': 'War'},
        'PEACE':          {'color': '#808080', 'style': 'dotted', 'label': 'Peace/Neutral'}
    }

    # Popolamento Grafo
    for _, row in trust_df.iterrows():
        u, v = row['observer_id'], row['target_id']
        if u == v: continue
        
        rel = row['relationship_state']
        cfg = REL_MAP.get(rel, REL_MAP['PEACE'])
        
        G.add_edge(u, v, 
                   color=cfg['color'], 
                   style=cfg['style'],
                   weight=(row['trust_value'] / 20.0) + 0.5)

    # Layout
    pos = nx.spring_layout(G, k=1.5, seed=42)
    
    # 1. Disegno Nodi
    nx.draw_networkx_nodes(G, pos, node_color='#2c3e50', node_size=4000, alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_color='white', font_size=10, font_weight='bold')

    # 2. Disegno Archi
    for style in ['solid', 'dashed', 'dotted']:
        edgelist = [(u, v) for u, v, d in G.edges(data=True) if d['style'] == style]
        if not edgelist: continue
        
        colors = [G[u][v]['color'] for u, v in edgelist]
        widths = [G[u][v]['weight'] for u, v in edgelist]
        
        nx.draw_networkx_edges(G, pos, edgelist=edgelist, edge_color=colors,
                               width=widths, style=style, arrows=True, 
                               arrowsize=25, connectionstyle='arc3,rad=0.1')

    # 3. Legenda dinamica
    legend_handles = [
        mlines.Line2D([], [], color=v['color'], linestyle=v['style'], label=v['label'])
        for k, v in REL_MAP.items()
    ]
    
    plt.title(f"GeoMAS: Mappa Diplomatica (Turno {max_turn})", pad=20, fontsize=15)
    plt.legend(handles=legend_handles, loc='upper right', frameon=True, shadow=True)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Nessun dato di Trust trovato.")